# Indego station-hour panel: join and cleaning

This notebook builds the analysis table for the Philadelphia bike-share demand project.
It does not fit models. It cleans Indego trips, rolls them up to station-hours, and
attaches DisPatch70 context plus two case-study calendars that DisPatch70 does not cover.

*Skip this step when using the supplied processed files.

## Colab

Run this notebook before 02_eda.ipynb, in the same runtime. A new runtime starts empty.


In [1]:
import sys
import json
import importlib.util
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")

    import subprocess
    for package, module in [('numpy', 'numpy'), ('pandas', 'pandas'), ('pyarrow', 'pyarrow')]:
        if importlib.util.find_spec(module) is None:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

if IN_COLAB:
    ROOT = Path("/content/drive/MyDrive/Milestone II")
else:
    ROOT = Path.cwd().resolve()

ROOT.mkdir(parents=True, exist_ok=True)

PROCESSED = ROOT / "data/processed"
SHARED = ROOT / "data/shared"

# Create shared project settings
settings_path = ROOT / "project_settings.json"

default_settings = {
    "study_start": "2024-01-01",
    "train_end": "2026-01-01",
    "validation_end": "2026-04-01",
    "test_end": "2026-07-01",
    "timezone": "America/New_York",
    "forecast_profiles_available_from": "2024-10-01",
    "forward_validation_windows": [
        ["2025-01-01", "2025-04-01"],
        ["2025-04-01", "2025-07-01"],
        ["2025-07-01", "2025-10-01"],
        ["2025-10-01", "2026-01-01"],
        ["2026-01-01", "2026-04-01"]
    ]
}

if not settings_path.exists():
    settings_path.write_text(json.dumps(default_settings, indent=2))
    print("Created:", settings_path)
else:
    print("Using existing:", settings_path)

settings = json.loads(settings_path.read_text())

import numpy as np
import pandas as pd

STUDY_START = pd.Timestamp(settings["study_start"])
TRAIN_END = pd.Timestamp(settings["train_end"])
VAL_END = pd.Timestamp(settings["validation_end"])
TEST_END = pd.Timestamp(settings["test_end"])
PROFILE_AVAILABLE_FROM = pd.Timestamp(settings["forecast_profiles_available_from"])
CV_WINDOWS = settings["forward_validation_windows"]

assert STUDY_START < PROFILE_AVAILABLE_FROM < TRAIN_END < VAL_END < TEST_END
assert all(PROFILE_AVAILABLE_FROM < pd.Timestamp(a) < pd.Timestamp(b) <= VAL_END
           for a, b in CV_WINDOWS)

if settings["timezone"] != "America/New_York":
    raise ValueError("The prepared timestamps use Philadelphia local time. Rebuild data before changing timezone.")

print("Project:", ROOT)
print(f"Training: {STUDY_START.date()} to {(TRAIN_END - pd.Timedelta(days=1)).date()}")
print(f"Validation: {TRAIN_END.date()} to {(VAL_END - pd.Timedelta(days=1)).date()}")
print(f"Test: {VAL_END.date()} to {(TEST_END - pd.Timedelta(days=1)).date()}")

Mounted at /content/drive
Using existing: /content/drive/MyDrive/Milestone II/project_settings.json
Project: /content/drive/MyDrive/Milestone II
Training: 2024-01-01 to 2025-12-31
Validation: 2026-01-01 to 2026-03-31
Test: 2026-04-01 to 2026-06-30


In [2]:
import json
import numpy as np
import pandas as pd
import re
import shutil
import urllib.error
import urllib.request
import zipfile
from pathlib import Path

def norm_name(value):
    """Ignore punctuation and spacing when comparing station names. Keep street numbers."""
    if pd.isna(value):
        return ""
    text = str(value).lower().replace("&", " and ")
    text = text.replace("m.l.k.", "martin luther king")
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def haversine_km(lat1, lon1, lat2, lon2):
    """Calculate straight-line distance between latitude/longitude pairs in kilometres."""
    lat1 = np.radians(np.asarray(lat1, dtype="float64"))
    lon1 = np.radians(np.asarray(lon1, dtype="float64"))
    lat2 = np.radians(np.asarray(lat2, dtype="float64"))
    lon2 = np.radians(np.asarray(lon2, dtype="float64"))
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    return 6371.0088 * 2.0 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


In [3]:

RAW = ROOT / "data/raw"

STUDY_YEARS = tuple(range(STUDY_START.year, (TEST_END - pd.Timedelta(days=1)).year + 1))
TZ = "America/New_York"
BBOX = {"lat": (39.80, 40.15), "lon": (-75.35, -74.90)}
VIRTUAL_STATION_ID = 3000
NEAR_SPORTS_KM = 1.5
# Dispatch sometimes uses a shorter name than the Indego station table.
NAME_ALIASES = {
    "philadelphia art museum": "philadelphia museum of art",
}
HF = "https://huggingface.co/datasets/hulnegy/DisPatch70/resolve/main"
INDEGO_PAGE = "https://www.rideindego.com/about/data/"
USER_AGENT = "Mozilla/5.0 (research; Indego-DisPatch70 join)"

# Stadium coordinates are the published locations of the venues named in DisPatch70.
# Xfinity Mobile Arena is the renamed Wells Fargo Center, so both names share a point.
VENUES = {
    "citizens bank park": (39.906057, -75.166495),
    "lincoln financial field": (39.900833, -75.167500),
    "xfinity mobile arena": (39.901201, -75.171979),
    "wells fargo center": (39.901201, -75.171979),
    "subaru park": (39.832778, -75.378611),
    "franklin field": (39.950190, -75.190000),
    "liacouras center": (39.980300, -75.158000),
}

# SEPTA's 20% Bus and Metro cut ran from August 24, 2025 through September 13, 2025
# and was restored September 14, 2025, when the 21.5% fare increase also took effect.
# City of Philadelphia: phila.gov/2025-08-19-city-response-septa-service-cuts/
SEPTA_CUT_START = pd.Timestamp("2025-08-24")
SEPTA_CUT_END = pd.Timestamp("2025-09-13 23:00:00")
SEPTA_FARE_START = pd.Timestamp("2025-09-14")

# Kickoffs from the City of Philadelphia Office of Special Events (June 10, 2026).
# The July 4 match is also the city's semiquincentennial celebration.
FIFA_MATCHES = [
    ("2026-06-14 19:00", "Cote d'Ivoire vs Ecuador"),
    ("2026-06-19 20:30", "Brazil vs Haiti"),
    ("2026-06-22 17:00", "France vs Iraq"),
    ("2026-06-25 16:00", "Curacao vs Cote d'Ivoire"),
    ("2026-06-27 17:00", "Croatia vs Ghana"),
    ("2026-07-04 17:00", "Round of 16"),
]

WEATHER_KIND_COLUMNS = {
    "rain": "wx_rain",
    "heavy_rain": "wx_heavy_rain",
    "heat": "wx_heat",
    "cold": "wx_cold",
    "snow": "wx_snow",
    "wind": "wx_wind",
    "fog": "wx_fog",
    "air_quality": "wx_poor_air",
}

SITUATION_MEAN = [
    "temperature_2m",
    "apparent_temperature",
    "relative_humidity_2m",
    "precipitation",
    "rain",
    "snowfall",
    "snow_depth",
    "weather_code",
    "cloud_cover",
    "wind_speed_10m",
    "wind_gusts_10m",
    "pm2_5",
    "us_aqi",
]
SITUATION_SUM = ["typical_trips"]
SITUATION_FIRST = [
    "is_holiday",
    "holiday_name",
    "day_type",
    "regime",
    "is_day",
    "weather_text",
]


def section(title: str) -> None:
    print(f"\n{'=' * 72}\n{title}\n{'=' * 72}")


def download(url: str, dest: Path) -> Path | None:
    """Download url to dest. Return None on HTTP 404. Reuse a non-empty local file."""
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    request = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
    try:
        with urllib.request.urlopen(request, timeout=180) as response, dest.open("wb") as handle:
            shutil.copyfileobj(response, handle)
    except urllib.error.HTTPError as exc:
        if dest.exists():
            dest.unlink()
        if exc.code == 404:
            return None
        raise
    if dest.stat().st_size == 0:
        dest.unlink()
        return None
    return dest


def fetch_text(url: str) -> str:
    request = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
    with urllib.request.urlopen(request, timeout=60) as response:
        return response.read().decode("utf-8", "replace")


def numeric_station_id(values: pd.Series) -> pd.Series:
    extracted = values.astype("string").str.extract(r"(\d+)$", expand=False)
    return pd.to_numeric(extracted, errors="coerce").astype("Int64")


def in_bbox(lat: pd.Series, lon: pd.Series) -> pd.Series:
    return lat.between(*BBOX["lat"]) & lon.between(*BBOX["lon"])


def as_local_naive(value) -> pd.Timestamp:
    stamp = pd.Timestamp(value)
    if stamp.tzinfo is not None:
        stamp = stamp.tz_convert(TZ).tz_localize(None)
    return stamp


def overlapping_hours(start, end) -> pd.DatetimeIndex:
    """Hours that overlap the half-open local interval [start, end)."""
    if pd.isna(start):
        return pd.DatetimeIndex([])
    start = as_local_naive(start)
    end = start if pd.isna(end) else as_local_naive(end)
    first = start.floor("h")
    if end <= start:
        return pd.DatetimeIndex([first])
    last = (end - pd.Timedelta(nanoseconds=1)).floor("h")
    if last < first:
        last = first
    return pd.date_range(first, last, freq="h")


def first_valid(series: pd.Series):
    kept = series.dropna()
    if kept.empty:
        return pd.NA
    return kept.iloc[0]

## Download

Indego quarters are discovered from the current data page, then limited to 2024-2026.
DisPatch70 files that have not been published yet (for example a future month of station
status) are skipped when the Hugging Face URL returns 404.


In [4]:
def indego_trip_urls() -> list[str]:
    html = fetch_text(INDEGO_PAGE)
    found = re.findall(
        r"https://www\.rideindego\.com/wp-content/uploads/\d{4}/\d{2}/indego-trips-20(?:24|25|26)-q[1-4]\.zip",
        html,
    )
    # The page lists newest first. Keep that order, but drop duplicates.
    seen = set()
    urls = []
    for url in found:
        if url not in seen:
            seen.add(url)
            urls.append(url)
    if not urls:
        raise RuntimeError("No 2024-2026 Indego trip links were found on the Indego data page.")
    return urls


def indego_station_url() -> str:
    html = fetch_text(INDEGO_PAGE)
    match = re.search(
        r"https://www\.rideindego\.com/wp-content/uploads/[^\"']+indego-stations-[^\"']+\.csv",
        html,
    )
    if not match:
        raise RuntimeError("The Indego station table link was not found.")
    return match.group(0)


def download_inputs() -> dict[str, list[Path]]:
    section("Downloading Indego trips, the station table, and Philadelphia DisPatch70 tables")
    trips = []
    for url in indego_trip_urls():
        dest = RAW / "indego" / Path(url).name
        saved = download(url, dest)
        print(f"  trips  {saved.name}")
        trips.append(saved)

    station_url = indego_station_url()
    stations = download(station_url, RAW / "indego" / Path(station_url).name)
    print(f"  stations  {stations.name}")

    context = download(
        f"{HF}/station_context/philadelphia/indego.parquet",
        RAW / "dispatch70" / "station_context.parquet",
    )
    print("  station context")

    buckets: dict[str, list[Path]] = {
        "trips": trips,
        "stations": [stations],
        "context": [context],
        "situations": [],
        "weather": [],
        "sports": [],
        "status": [],
        "dispatch": [],
    }
    months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
    for year in STUDY_YEARS:
        for kind, url, dest in [
            ("situations", f"{HF}/situations/philadelphia/{year}.parquet", RAW / "dispatch70" / "situations" / f"{year}.parquet"),
            ("weather", f"{HF}/events/philadelphia/weather/{year}.parquet", RAW / "dispatch70" / "weather" / f"{year}.parquet"),
            ("sports", f"{HF}/events/philadelphia/sports_espn/{year}.parquet", RAW / "dispatch70" / "sports" / f"{year}.parquet"),
            ("dispatch", f"{HF}/dispatch/philadelphia/{year}.parquet", RAW / "dispatch70" / "dispatch" / f"{year}.parquet"),
        ]:
            saved = download(url, dest)
            if saved is None:
                print(f"  missing {kind} {year}")
            else:
                buckets[kind].append(saved)
                print(f"  {kind} {year}")
        for month in months:
            saved = download(
                f"{HF}/station_status/philadelphia/indego/{year}/{month}.parquet",
                RAW / "dispatch70" / "status" / str(year) / f"{month}.parquet",
            )
            if saved is not None:
                buckets["status"].append(saved)
        print(f"  station status {year}: {sum(1 for p in buckets['status'] if p.parent.name == str(year))} months")
    return buckets

## Clean Indego trips

Quarterly files are concatenated and checked against the published layout
(`trip_id`, times, stations, coordinates, duration, route category, pass type, bike type).
Times are local Philadelphia time with no timezone offset in the file.

Rows removed:

- unparseable start time, missing start station, or a duplicate `trip_id` + start time
- duration outside 1 to 1,440 minutes (Indego's own published bounds)
- trips that start at Virtual Station (`3000`), which is a staff checkout bucket
- start coordinates that are missing or outside the Philadelphia operating area

Trips that end at Virtual Station, or whose end coordinates are missing, still count as
departures. They do not count as arrivals at a real station.


In [5]:
REQUIRED_TRIP_COLUMNS = [
    "trip_id",
    "duration",
    "start_time",
    "end_time",
    "start_station",
    "start_lat",
    "start_lon",
    "end_station",
    "end_lat",
    "end_lon",
    "trip_route_category",
    "passholder_type",
    "bike_type",
]


def read_trip_zip(path: Path) -> pd.DataFrame:
    with zipfile.ZipFile(path) as archive:
        members = [
            name
            for name in archive.namelist()
            if name.lower().endswith(".csv") and not name.startswith("__MACOSX") and "/." not in name
        ]
        if len(members) != 1:
            raise RuntimeError(f"{path.name} contains {len(members)} CSV files: {members}")
        with archive.open(members[0]) as handle:
            frame = pd.read_csv(handle, low_memory=False)
    frame.columns = [str(column).strip().lower() for column in frame.columns]
    missing = [column for column in REQUIRED_TRIP_COLUMNS if column not in frame.columns]
    if missing:
        raise RuntimeError(f"{path.name} is missing columns {missing}")
    keep = REQUIRED_TRIP_COLUMNS + (["plan_duration"] if "plan_duration" in frame.columns else [])
    frame = frame[keep].copy()
    frame["source_file"] = path.name
    return frame


def clean_trips(frames: list[pd.DataFrame]) -> tuple[pd.DataFrame, pd.DataFrame]:
    raw = pd.concat(frames, ignore_index=True)
    report_rows = [("rows_read", len(raw))]
    trips = raw

    trips["start_time"] = pd.to_datetime(trips["start_time"], format="mixed", errors="coerce")
    trips["end_time"] = pd.to_datetime(trips["end_time"], format="mixed", errors="coerce")
    trips["duration_min"] = pd.to_numeric(trips["duration"], errors="coerce")
    trips["start_station_id"] = numeric_station_id(trips["start_station"])
    trips["end_station_id"] = numeric_station_id(trips["end_station"])
    for column in ("start_lat", "start_lon", "end_lat", "end_lon", "plan_duration"):
        if column in trips.columns:
            trips[column] = pd.to_numeric(trips[column], errors="coerce")

    def drop(mask: pd.Series, reason: str) -> None:
        nonlocal trips
        report_rows.append((reason, int(mask.fillna(False).sum())))
        trips = trips.loc[~mask.fillna(False)].copy()

    drop(trips["start_time"].isna(), "dropped_bad_start_time")
    drop(trips["start_station_id"].isna(), "dropped_missing_start_station")
    drop(trips.duplicated(["trip_id", "start_time"], keep="first"), "dropped_duplicate_trip")
    drop(~trips["duration_min"].between(1, 1440), "dropped_duration_out_of_bounds")
    drop(trips["start_station_id"] == VIRTUAL_STATION_ID, "dropped_virtual_start")
    drop(~in_bbox(trips["start_lat"], trips["start_lon"]), "dropped_start_outside_philadelphia")

    # A virtual or unusable end station is not an arrival. The departure still stands.
    bad_end = (trips["end_station_id"] == VIRTUAL_STATION_ID) | trips["end_station_id"].isna()
    report_rows.append(("departures_with_no_real_end_station", int(bad_end.sum())))
    trips.loc[bad_end, "end_station_id"] = pd.NA

    good_end_coords = in_bbox(trips["end_lat"], trips["end_lon"])
    distance = np.full(len(trips), np.nan)
    if len(trips):
        distance = haversine_km(trips["start_lat"], trips["start_lon"], trips["end_lat"], trips["end_lon"])
        distance = np.where(good_end_coords.to_numpy(), distance, np.nan)
    trips["distance_km"] = distance
    trips["hour"] = trips["start_time"].dt.floor("h")
    trips["end_hour"] = trips["end_time"].dt.floor("h")
    trips["is_electric"] = trips["bike_type"].astype("string").str.lower().str.contains("electric", na=False)
    trips["is_one_way"] = trips["trip_route_category"].astype("string").str.lower().eq("one way")
    trips["passholder_type"] = trips["passholder_type"].astype("string")
    trips["bike_type"] = trips["bike_type"].astype("string").str.lower()

    window = trips["start_time"].dt.year.between(STUDY_YEARS[0], STUDY_YEARS[-1])
    drop(~window, "dropped_outside_study_years")
    report_rows.append(("rows_kept", len(trips)))
    report = pd.DataFrame(report_rows, columns=["step", "rows"])
    return trips, report

## Stations

The official station table supplies the name, go-live date, and whether Indego still lists
the station as active. DisPatch70 station context supplies coordinates, capacity, district,
nearby transit, and OpenStreetMap place counts. Trip coordinates fill in only if context
has no point for that id.

A station stays in the panel from its go-live date (or the start of the trip window)
through the end of the window if it is still active. Inactive stations end at their last
observed trip, because the station table does not publish a closure timestamp. Hours before
go-live are not filled with zeros.


In [6]:
def parse_json_object(value) -> dict:
    if not isinstance(value, str) or not value:
        return {}
    try:
        parsed = json.loads(value)
    except json.JSONDecodeError:
        return {}
    return parsed if isinstance(parsed, dict) else {}


def parse_json_list(value) -> list:
    if not isinstance(value, str) or not value:
        return []
    try:
        parsed = json.loads(value)
    except json.JSONDecodeError:
        return []
    return parsed if isinstance(parsed, list) else []


def build_stations(station_csv: Path, context_path: Path, trips: pd.DataFrame) -> pd.DataFrame:
    official = pd.read_csv(station_csv, encoding="utf-8-sig")
    official.columns = [re.sub(r"\s+", " ", str(column)).strip() for column in official.columns]
    official = official.rename(
        columns={
            "Station_ID": "station_id",
            "Station_Name": "station_name",
            "Day of Go_live_date": "go_live",
            "Status": "status",
        }
    )
    official["station_id"] = pd.to_numeric(official["station_id"], errors="coerce").astype("Int64")
    official["go_live"] = pd.to_datetime(official["go_live"], format="mixed", errors="coerce").dt.normalize()
    official["station_name"] = official["station_name"].astype("string").str.strip()
    official["status"] = official["status"].astype("string").str.strip()

    context = pd.read_parquet(context_path)
    context["station_id"] = numeric_station_id(context["station_id"])
    poi = pd.json_normalize(context["poi_counts"].map(parse_json_object)).fillna(0)
    poi.index = context.index
    context = pd.concat([context, poi.add_prefix("poi_")], axis=1)

    def transit_features(value) -> pd.Series:
        items = parse_json_list(value)
        distances = [item.get("distance_m") for item in items if isinstance(item, dict) and item.get("distance_m") is not None]
        return pd.Series(
            {
                "n_transit": len(items),
                "nearest_transit_m": min(distances) if distances else np.nan,
            }
        )

    def neighbor_features(value) -> pd.Series:
        items = parse_json_list(value)
        distances = [item.get("distance_m") for item in items if isinstance(item, dict) and item.get("distance_m") is not None]
        return pd.Series(
            {
                "nearest_station_m": min(distances) if distances else np.nan,
                "n_stations_within_500m": sum(distance <= 500 for distance in distances),
            }
        )

    transit = context["transit"].apply(transit_features)
    neighbors = context["neighbour_stations"].apply(neighbor_features)
    context_keep = pd.concat(
        [
            context[
                [
                    "station_id",
                    "name",
                    "capacity",
                    "lat",
                    "lon",
                    "street",
                    "neighbourhood",
                    "district",
                ]
            ],
            poi[
                [column for column in ["cafe", "restaurant", "bar_pub", "park", "university", "office", "shop", "hotel"] if column in poi.columns]
            ].add_prefix("poi_"),
            transit,
            neighbors,
        ],
        axis=1,
    )

    trip_coords = (
        trips.groupby("start_station_id")[["start_lat", "start_lon"]]
        .median()
        .rename(columns={"start_lat": "trip_lat", "start_lon": "trip_lon"})
    )
    first_start = trips.groupby("start_station_id")["hour"].min()
    last_start = trips.groupby("start_station_id")["hour"].max()
    first_end = trips.dropna(subset=["end_station_id"]).groupby("end_station_id")["end_hour"].min()
    last_end = trips.dropna(subset=["end_station_id"]).groupby("end_station_id")["end_hour"].max()
    first_seen = pd.concat([first_start, first_end], axis=1).min(axis=1).rename("first_seen_hour")
    last_seen = pd.concat([last_start, last_end], axis=1).max(axis=1).rename("last_seen_hour")

    stations = official.merge(context_keep, on="station_id", how="outer")
    stations = stations.merge(trip_coords, left_on="station_id", right_index=True, how="outer")
    stations = stations.merge(first_seen, left_on="station_id", right_index=True, how="left")
    stations = stations.merge(last_seen, left_on="station_id", right_index=True, how="left")
    stations["station_name"] = stations["station_name"].fillna(stations["name"])
    stations["station_name"] = stations["station_name"].fillna("Station " + stations["station_id"].astype("string"))
    stations["lat"] = stations["lat"].fillna(stations["trip_lat"])
    stations["lon"] = stations["lon"].fillna(stations["trip_lon"])
    stations["norm_name"] = stations["station_name"].map(norm_name)
    stations = stations.loc[stations["station_id"].notna() & (stations["station_id"] != VIRTUAL_STATION_ID)].copy()
    stations = stations.drop_duplicates("station_id")
    return stations


def service_bounds(stations: pd.DataFrame, window_start: pd.Timestamp, window_end: pd.Timestamp) -> pd.DataFrame:
    """Return stations that were open during the trip window, with start and end hours."""
    rows = []
    for station in stations.itertuples(index=False):
        if pd.isna(station.go_live):
            start = window_start
        else:
            start = max(pd.Timestamp(station.go_live).floor("h"), window_start)
        # A trip before the published go-live date is real; don't leave it outside the panel.
        if pd.notna(station.first_seen_hour) and station.first_seen_hour < start:
            start = max(pd.Timestamp(station.first_seen_hour).floor("h"), window_start)
        if str(station.status) == "Active" or pd.isna(station.status):
            end = window_end
        elif pd.notna(station.last_seen_hour):
            end = min(pd.Timestamp(station.last_seen_hour), window_end)
        else:
            continue
        if start > end or start > window_end:
            continue
        # A station with no activity and no official active flag is not inventoried.
        if pd.isna(station.status) and pd.isna(station.last_seen_hour):
            continue
        if str(station.status) == "Inactive" and (pd.isna(station.last_seen_hour) or station.last_seen_hour < window_start):
            continue
        rows.append((int(station.station_id), start.floor("h"), end.floor("h")))
    bounds = pd.DataFrame(rows, columns=["station_id", "service_start", "service_end"])
    return bounds

## Station-hour trips

Departures are counted at the start station and start hour. Arrivals are counted at the
end station and end hour. Mix features (duration, distance, electric share, one-way share)
describe departures only, and they stay missing in hours with no departures rather than
being filled with zero.


In [7]:
def aggregate_trips(trips: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    departures = (
        trips.groupby(["start_station_id", "hour"], as_index=False)
        .agg(
            departures=("trip_id", "size"),
            duration_median_min=("duration_min", "median"),
            distance_median_km=("distance_km", "median"),
            electric_share=("is_electric", "mean"),
            one_way_share=("is_one_way", "mean"),
        )
        .rename(columns={"start_station_id": "station_id"})
    )
    arrivals = (
        trips.dropna(subset=["end_station_id", "end_hour"])
        .groupby(["end_station_id", "end_hour"], as_index=False)
        .size()
        .rename(columns={"end_station_id": "station_id", "end_hour": "hour", "size": "arrivals"})
    )
    return departures, arrivals


def build_station_hour_index(bounds: pd.DataFrame) -> pd.DataFrame:
    ids = []
    hours = []
    for station_id, start, end in bounds.itertuples(index=False):
        stamp = pd.date_range(start, end, freq="h")
        ids.append(np.full(len(stamp), station_id, dtype=np.int32))
        hours.append(stamp.to_numpy())
    return pd.DataFrame(
        {
            "station_id": np.concatenate(ids),
            "hour": pd.to_datetime(np.concatenate(hours)),
        }
    )

## DisPatch70 context

Situations are 30-minute city snapshots. Within each clock hour the two precipitation
readings match, so weather is averaged. typical_trips is a different 30-minute profile
slot, so the two slots are summed to an hour. That column is DisPatch70's usual volume
for the time of day, not the observed Indego count.

Weather-event rows are intervals (snow, heat, etc.), expanded onto the hours they
overlap. Sports rows have a venue name and no coordinates; distance uses the stadium
lookup above and keeps the nearest overlapping event. Postponed games are excluded.
Station status snapshots are averaged inside each station-hour. Hours with no snapshot
stay missing, because "no reading" is not "zero bikes."

Dispatch events use station names. Names are normalized, then unmatched names are paired
with a unique close match when one exists. Moves add bikes at the destination and remove
them at the origin.


In [8]:
def load_situations(paths: list[Path]) -> pd.DataFrame:
    frames = [pd.read_parquet(path) for path in paths]
    raw = pd.concat(frames, ignore_index=True)
    raw["hour"] = pd.to_datetime(raw["ts_local"]).dt.floor("h")
    means = raw.groupby("hour", as_index=False)[SITUATION_MEAN].mean()
    sums = raw.groupby("hour", as_index=False)[SITUATION_SUM].sum()
    top_of_hour = raw.sort_values("ts_local").groupby("hour", as_index=False)[SITUATION_FIRST].agg(first_valid)
    situations = means.merge(sums, on="hour").merge(top_of_hour, on="hour")
    situations = situations.rename(
        columns={
            "temperature_2m": "temperature_c",
            "apparent_temperature": "apparent_temperature_c",
            "relative_humidity_2m": "humidity",
            "precipitation": "precipitation_mm",
            "rain": "rain_mm",
            "snowfall": "snowfall_cm",
            "wind_speed_10m": "wind_speed_kmh",
            "wind_gusts_10m": "wind_gusts_kmh",
            "typical_trips": "typical_city_departures",
        }
    )
    situations["is_holiday"] = situations["is_holiday"].astype("boolean")
    situations["is_day"] = situations["is_day"].astype("boolean")
    return situations


def load_weather_flags(paths: list[Path]) -> tuple[pd.DataFrame, set[int]]:
    columns = list(WEATHER_KIND_COLUMNS.values())
    if not paths:
        return pd.DataFrame(columns=["hour", *columns]), set()
    frames = [pd.read_parquet(path, columns=["src_kind", "start_local", "end_local"]) for path in paths]
    events = pd.concat(frames, ignore_index=True)
    years = set()
    records = []
    for event in events.itertuples(index=False):
        column = WEATHER_KIND_COLUMNS.get(str(event.src_kind))
        if column is None:
            continue
        for hour in overlapping_hours(event.start_local, event.end_local):
            years.add(int(hour.year))
            records.append((hour, column))
    if not records:
        flags = pd.DataFrame(columns=["hour", *columns])
    else:
        long = pd.DataFrame(records, columns=["hour", "flag"])
        flags = (
            long.assign(value=True)
            .pivot_table(index="hour", columns="flag", values="value", aggfunc="any", fill_value=False)
            .reset_index()
        )
    for column in columns:
        if column not in flags.columns:
            flags[column] = False
    # Years come from the files even when a kind never occurs.
    for path in paths:
        years.add(int(path.stem))
    return flags[["hour", *columns]], years


def load_sports(paths: list[Path]) -> tuple[pd.DataFrame, pd.DataFrame]:
    empty_city = pd.DataFrame(columns=["hour", "sports_any", "sports_attendance_max"])
    empty_near = pd.DataFrame(columns=["hour", "station_id", "sports_league", "sports_attendance", "km_to_sports"])
    if not paths:
        return empty_city, empty_near
    sports = pd.concat([pd.read_parquet(path) for path in paths], ignore_index=True)
    sports = sports.loc[sports["status"].astype("string").eq("STATUS_FINAL")].copy()
    sports["venue_key"] = sports["venue_name"].astype("string").str.lower().str.strip()
    unknown = sorted(set(sports["venue_key"]) - set(VENUES))
    if unknown:
        print("  sports venues without coordinates:", ", ".join(unknown))
    sports["venue_lat"] = sports["venue_key"].map(lambda key: VENUES.get(key, (np.nan, np.nan))[0])
    sports["venue_lon"] = sports["venue_key"].map(lambda key: VENUES.get(key, (np.nan, np.nan))[1])
    hour_rows = []
    for event in sports.itertuples(index=False):
        for hour in overlapping_hours(event.start_local, event.end_local):
            hour_rows.append(
                (
                    hour,
                    event.src_league,
                    event.expected_attendance,
                    event.venue_lat,
                    event.venue_lon,
                )
            )
    if not hour_rows:
        return empty_city, empty_near
    expanded = pd.DataFrame(
        hour_rows, columns=["hour", "sports_league", "sports_attendance", "venue_lat", "venue_lon"]
    )
    city = expanded.groupby("hour", as_index=False).agg(
        sports_any=("sports_league", "size"),
        sports_attendance_max=("sports_attendance", "max"),
    )
    city["sports_any"] = city["sports_any"] > 0
    return city, expanded


def nearest_sports(expanded: pd.DataFrame, stations: pd.DataFrame) -> pd.DataFrame:
    if expanded.empty or "venue_lat" not in expanded.columns:
        return pd.DataFrame(columns=["hour", "station_id", "sports_league", "sports_attendance", "km_to_sports"])
    located = stations.loc[stations["lat"].notna() & stations["lon"].notna(), ["station_id", "lat", "lon"]]
    events = expanded.loc[expanded["venue_lat"].notna() & expanded["venue_lon"].notna()].copy()
    if located.empty or events.empty:
        return pd.DataFrame(columns=["hour", "station_id", "sports_league", "sports_attendance", "km_to_sports"])
    pairs = events.merge(located, how="cross")
    pairs["km_to_sports"] = haversine_km(pairs["lat"], pairs["lon"], pairs["venue_lat"], pairs["venue_lon"])
    nearest_index = pairs.groupby(["station_id", "hour"])["km_to_sports"].idxmin()
    nearest = pairs.loc[nearest_index, ["station_id", "hour", "sports_league", "sports_attendance", "km_to_sports"]]
    return nearest.reset_index(drop=True)


def load_status(paths: list[Path]) -> pd.DataFrame:
    if not paths:
        return pd.DataFrame(
            columns=["station_id", "hour", "bikes_available", "docks_available", "zero_bikes", "full_station", "n_status_obs"]
        )
    frames = [
        pd.read_parquet(path, columns=["commit_at", "bikes", "stands", "station_id"])
        for path in paths
    ]
    status = pd.concat(frames, ignore_index=True)
    local = pd.to_datetime(status["commit_at"], utc=True).dt.tz_convert(TZ).dt.tz_localize(None)
    status["hour"] = local.dt.floor("h")
    status["station_id"] = numeric_station_id(status["station_id"])
    status = status.dropna(subset=["station_id"])
    grouped = status.groupby(["station_id", "hour"], as_index=False).agg(
        bikes_available=("bikes", "mean"),
        docks_available=("stands", "mean"),
        bikes_min=("bikes", "min"),
        docks_min=("stands", "min"),
        n_status_obs=("bikes", "size"),
    )
    grouped["zero_bikes"] = grouped["bikes_min"].eq(0)
    grouped["full_station"] = grouped["docks_min"].eq(0)
    return grouped.drop(columns=["bikes_min", "docks_min"])


def load_dispatch(paths: list[Path], stations: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, set[int]]:
    columns = ["station_id", "hour", "rebalance_bikes_in", "rebalance_bikes_out", "service_events"]
    if not paths:
        return pd.DataFrame(columns=columns), pd.DataFrame(), set()
    dispatch = pd.concat([pd.read_parquet(path) for path in paths], ignore_index=True)
    years = {int(path.stem) for path in paths}
    name_to_id = (
        stations.dropna(subset=["norm_name"])
        .drop_duplicates("norm_name")
        .set_index("norm_name")["station_id"]
        .to_dict()
    )
    choices = list(name_to_id)
    raw_names = pd.unique(
        pd.concat([dispatch["from_station"], dispatch["to_station"]], ignore_index=True).dropna()
    )
    match_rows = []
    resolved = dict(name_to_id)
    for name in raw_names:
        key = norm_name(name)
        lookup = NAME_ALIASES.get(key, key)
        station_id = name_to_id.get(lookup)
        if key in NAME_ALIASES and station_id is not None:
            method = "alias"
        else:
            method = "normalized"
        if station_id is None and lookup:
            import difflib

            close = difflib.get_close_matches(lookup, choices, n=1, cutoff=0.90)
            if close:
                station_id = name_to_id[close[0]]
                method = f"fuzzy:{close[0]}"
        if station_id is not None:
            resolved[key] = int(station_id)
            resolved[lookup] = int(station_id)
        match_rows.append((name, key, station_id, method if station_id is not None else "unmatched"))
    matches = pd.DataFrame(match_rows, columns=["dispatch_name", "normalized", "station_id", "method"])

    timestamp = dispatch["t_from"].fillna(dispatch["t_to"])
    local = pd.to_datetime(timestamp, unit="s", utc=True).dt.tz_convert(TZ).dt.tz_localize(None)
    dispatch = dispatch.assign(hour=local.dt.floor("h"))
    dispatch["from_id"] = dispatch["from_station"].map(norm_name).map(resolved)
    dispatch["to_id"] = dispatch["to_station"].map(norm_name).map(resolved)

    moves = dispatch.loc[dispatch["kind"].eq("move")]
    inbound = moves.dropna(subset=["to_id"]).groupby(["to_id", "hour"], as_index=False)["n_bikes"].sum()
    inbound = inbound.rename(columns={"to_id": "station_id", "n_bikes": "rebalance_bikes_in"})
    outbound = moves.dropna(subset=["from_id"]).groupby(["from_id", "hour"], as_index=False)["n_bikes"].sum()
    outbound = outbound.rename(columns={"from_id": "station_id", "n_bikes": "rebalance_bikes_out"})
    service = dispatch.loc[dispatch["kind"].eq("service")].dropna(subset=["from_id"])
    service = service.groupby(["from_id", "hour"], as_index=False).size()
    service = service.rename(columns={"from_id": "station_id", "size": "service_events"})

    combined = inbound.merge(outbound, on=["station_id", "hour"], how="outer").merge(
        service, on=["station_id", "hour"], how="outer"
    )
    for column in ("rebalance_bikes_in", "rebalance_bikes_out", "service_events"):
        combined[column] = combined[column].fillna(0)
    combined["station_id"] = combined["station_id"].astype("int32")
    return combined, matches, years

## Case-study flags and lags

SEPTA and World Cup dates are citywide time flags. They are not estimated from the trip
data. Recent-ridership lags are shifts on the same station. A lag is kept only when the
earlier row is the expected number of hours away, so a station that enters service mid-window
does not inherit another station's last hour.

departures_next_hour is the supervised target: departures from this station in the next hour.


In [9]:
def add_case_flags(panel: pd.DataFrame) -> pd.DataFrame:
    hour = panel["hour"]
    panel["is_septa_service_cut"] = hour.between(SEPTA_CUT_START, SEPTA_CUT_END)
    panel["is_septa_fare_increase"] = hour.ge(SEPTA_FARE_START)
    panel["is_semiquincentennial"] = hour.dt.normalize().eq(pd.Timestamp("2026-07-04"))

    match_hours = set()
    match_days = set()
    for kickoff, _label in FIFA_MATCHES:
        start = pd.Timestamp(kickoff)
        # Include the kickoff hour and the following three hours (through the match and the exit).
        for stamp in pd.date_range(start.floor("h"), (start + pd.Timedelta(hours=3)).floor("h"), freq="h"):
            match_hours.add(stamp)
        match_days.add(start.normalize())
    panel["is_fifa_match_hour"] = hour.isin(match_hours)
    panel["is_fifa_match_day"] = hour.dt.normalize().isin(match_days)
    return panel


def add_lags(panel: pd.DataFrame) -> pd.DataFrame:
    panel = panel.sort_values(["station_id", "hour"], kind="mergesort")
    grouped = panel.groupby("station_id", sort=False)
    panel["departures_next_hour"] = grouped["departures"].shift(-1)
    panel["departures_lag_1"] = grouped["departures"].shift(1)
    panel["departures_lag_24"] = grouped["departures"].shift(24)
    panel["departures_lag_168"] = grouped["departures"].shift(168)
    checks = {
        "departures_next_hour": (grouped["hour"].shift(-1), 1),
        "departures_lag_1": (grouped["hour"].shift(1), -1),
        "departures_lag_24": (grouped["hour"].shift(24), -24),
        "departures_lag_168": (grouped["hour"].shift(168), -168),
    }
    for column, (neighbor, hour_delta) in checks.items():
        expected = panel["hour"] + pd.Timedelta(hours=hour_delta)
        panel.loc[neighbor.ne(expected) | neighbor.isna(), column] = np.nan
    return panel


def fill_observed_zeros(panel: pd.DataFrame, columns: list[str], years: set[int]) -> pd.DataFrame:
    """Fill structural zeros only in years whose source file was actually loaded."""
    if not years:
        return panel
    observed = panel["hour"].dt.year.isin(years)
    for column in columns:
        if column not in panel.columns:
            panel[column] = np.nan
        panel.loc[observed, column] = panel.loc[observed, column].fillna(0)
    return panel

## Assemble and save

The panel is written to data/processed/station_hour.parquet. Cleaned trips and the station
table are saved beside it. A short coverage table records, for each month, how often weather
and station status actually joined.


In [10]:
PANEL_COLUMNS = [
    "station_id",
    "hour",
    "departures",
    "arrivals",
    "net_flow",
    "departures_next_hour",
    "departures_lag_1",
    "departures_lag_24",
    "departures_lag_168",
    "duration_median_min",
    "distance_median_km",
    "electric_share",
    "one_way_share",
    "hour_of_day",
    "day_of_week",
    "month",
    "is_weekend",
    "temperature_c",
    "apparent_temperature_c",
    "humidity",
    "precipitation_mm",
    "rain_mm",
    "snowfall_cm",
    "weather_code",
    "weather_text",
    "cloud_cover",
    "wind_speed_kmh",
    "wind_gusts_kmh",
    "pm2_5",
    "us_aqi",
    "is_holiday",
    "holiday_name",
    "day_type",
    "regime",
    "is_day",
    "typical_city_departures",
    "wx_rain",
    "wx_heavy_rain",
    "wx_heat",
    "wx_cold",
    "wx_snow",
    "wx_wind",
    "wx_fog",
    "wx_poor_air",
    "sports_any",
    "sports_league",
    "sports_attendance",
    "km_to_sports",
    "near_sports",
    "bikes_available",
    "docks_available",
    "zero_bikes",
    "full_station",
    "n_status_obs",
    "rebalance_bikes_in",
    "rebalance_bikes_out",
    "service_events",
    "is_septa_service_cut",
    "is_septa_fare_increase",
    "is_fifa_match_day",
    "is_fifa_match_hour",
    "is_semiquincentennial",
]


def assert_unique(frame: pd.DataFrame, keys: list[str], label: str) -> None:
    duplicate_count = int(frame.duplicated(keys).sum())
    if duplicate_count:
        raise RuntimeError(f"{label} has {duplicate_count} duplicate rows on {keys}")


def assemble_panel(
    index: pd.DataFrame,
    departures: pd.DataFrame,
    arrivals: pd.DataFrame,
    situations: pd.DataFrame,
    weather: pd.DataFrame,
    weather_years: set[int],
    sports_city: pd.DataFrame,
    sports_nearest: pd.DataFrame,
    status: pd.DataFrame,
    dispatch: pd.DataFrame,
    dispatch_years: set[int],
) -> pd.DataFrame:
    for frame, keys, label in [
        (departures, ["station_id", "hour"], "departures"),
        (arrivals, ["station_id", "hour"], "arrivals"),
        (situations, ["hour"], "situations"),
        (weather, ["hour"], "weather flags"),
        (sports_city, ["hour"], "sports"),
        (sports_nearest, ["station_id", "hour"], "nearest sports"),
        (status, ["station_id", "hour"], "station status"),
        (dispatch, ["station_id", "hour"], "dispatch"),
    ]:
        if len(frame):
            assert_unique(frame, keys, label)

    panel = index.merge(departures, on=["station_id", "hour"], how="left")
    panel = panel.merge(arrivals, on=["station_id", "hour"], how="left")
    panel["departures"] = panel["departures"].fillna(0)
    panel["arrivals"] = panel["arrivals"].fillna(0)
    panel["net_flow"] = panel["arrivals"] - panel["departures"]

    panel = panel.merge(situations, on="hour", how="left")
    panel = panel.merge(weather, on="hour", how="left")
    for column in WEATHER_KIND_COLUMNS.values():
        if column not in panel.columns:
            panel[column] = np.nan
        panel[column] = pd.to_numeric(panel[column], errors="coerce")
    panel = fill_observed_zeros(panel, list(WEATHER_KIND_COLUMNS.values()), weather_years)
    for column in WEATHER_KIND_COLUMNS.values():
        panel[column] = panel[column].astype("boolean")

    if len(sports_city):
        panel = panel.merge(sports_city.drop(columns=["sports_attendance_max"]), on="hour", how="left")
    else:
        panel["sports_any"] = pd.NA
    panel["sports_any"] = panel["sports_any"].fillna(False).astype("boolean")
    panel = panel.merge(sports_nearest, on=["station_id", "hour"], how="left")
    panel["near_sports"] = pd.Series(pd.NA, index=panel.index, dtype="boolean")
    known_distance = panel["km_to_sports"].notna()
    panel.loc[known_distance, "near_sports"] = panel.loc[known_distance, "km_to_sports"] <= NEAR_SPORTS_KM
    panel.loc[~panel["sports_any"].fillna(False), "near_sports"] = False

    panel = panel.merge(status, on=["station_id", "hour"], how="left")
    panel = panel.merge(dispatch, on=["station_id", "hour"], how="left")
    panel = fill_observed_zeros(
        panel,
        ["rebalance_bikes_in", "rebalance_bikes_out", "service_events"],
        dispatch_years,
    )

    panel["hour_of_day"] = panel["hour"].dt.hour.astype("int8")
    panel["day_of_week"] = panel["hour"].dt.dayofweek.astype("int8")
    panel["month"] = panel["hour"].dt.month.astype("int8")
    panel["is_weekend"] = panel["day_of_week"].ge(5)
    panel = add_case_flags(panel)
    panel = add_lags(panel)

    for column in ("zero_bikes", "full_station", "is_holiday", "is_day"):
        if column in panel.columns:
            panel[column] = panel[column].astype("boolean")
    for column in (
        "is_septa_service_cut",
        "is_septa_fare_increase",
        "is_fifa_match_day",
        "is_fifa_match_hour",
        "is_semiquincentennial",
        "is_weekend",
    ):
        panel[column] = panel[column].astype("boolean")

    missing = [column for column in PANEL_COLUMNS if column not in panel.columns]
    if missing:
        raise RuntimeError(f"Panel is missing columns: {missing}")
    panel = panel[PANEL_COLUMNS]
    assert_unique(panel, ["station_id", "hour"], "panel")
    return panel


def coverage_table(panel: pd.DataFrame) -> pd.DataFrame:
    frame = panel.copy()
    frame["year_month"] = frame["hour"].dt.to_period("M").astype(str)
    grouped = frame.groupby("year_month", as_index=False).agg(
        station_hours=("station_id", "size"),
        departures=("departures", "sum"),
        stations=("station_id", "nunique"),
        weather_coverage=("temperature_c", lambda series: float(series.notna().mean())),
        status_coverage=("bikes_available", lambda series: float(series.notna().mean())),
        station_hours_with_sports=("sports_any", lambda series: int(series.fillna(False).sum())),
    )
    return grouped


def main() -> None:
    RAW.mkdir(parents=True, exist_ok=True)
    PROCESSED.mkdir(parents=True, exist_ok=True)
    print("Project:", ROOT)
    inputs = download_inputs()

    section("Reading and cleaning Indego trips")
    trip_frames = [read_trip_zip(path) for path in inputs["trips"]]
    trips, cleaning = clean_trips(trip_frames)
    del trip_frames
    print(cleaning.to_string(index=False))
    print(
        f"Kept trips from {trips['start_time'].min()} to {trips['start_time'].max()} "
        f"across {trips['start_station_id'].nunique()} start stations."
    )

    section("Building the station table and the station-hour index")
    stations = build_stations(inputs["stations"][0], inputs["context"][0], trips)
    window_start = trips["hour"].min()
    window_end = trips["hour"].max()
    bounds = service_bounds(stations, window_start, window_end)
    print(f"Stations in service during the window: {len(bounds):,}")
    print(f"Station-hour index will run from {window_start} to {window_end}.")
    departures, arrivals = aggregate_trips(trips)
    index = build_station_hour_index(bounds)
    print(f"In-service station-hours: {len(index):,}")

    section("Joining DisPatch70 and case-study calendars")
    situations = load_situations(inputs["situations"])
    weather, weather_years = load_weather_flags(inputs["weather"])
    sports_city, sports_expanded = load_sports(inputs["sports"])
    sports_nearest = nearest_sports(sports_expanded, stations)
    status = load_status(inputs["status"])
    dispatch, name_matches, dispatch_years = load_dispatch(inputs["dispatch"], stations)
    unmatched = name_matches.loc[name_matches["method"].eq("unmatched")]
    print(f"Dispatch names: {len(name_matches):,}; unmatched: {len(unmatched):,}")
    if len(unmatched):
        print(unmatched["dispatch_name"].head(20).to_string(index=False))

    panel = assemble_panel(
        index,
        departures,
        arrivals,
        situations,
        weather,
        weather_years,
        sports_city,
        sports_nearest,
        status,
        dispatch,
        dispatch_years,
    )
    coverage = coverage_table(panel)

    section("Saving tables")
    trip_out = trips[
        [
            "trip_id",
            "duration_min",
            "start_time",
            "end_time",
            "start_station_id",
            "end_station_id",
            "start_lat",
            "start_lon",
            "end_lat",
            "end_lon",
            "distance_km",
            "trip_route_category",
            "passholder_type",
            "bike_type",
            "plan_duration",
            "hour",
            "source_file",
        ]
    ].copy()
    trip_out["start_station_id"] = trip_out["start_station_id"].astype("int32")
    trip_path = PROCESSED / "trips_clean.parquet"
    station_path = PROCESSED / "stations.parquet"
    panel_path = PROCESSED / "station_hour.parquet"
    trip_out.to_parquet(trip_path, index=False)
    stations.drop(columns=["trip_lat", "trip_lon"], errors="ignore").to_parquet(station_path, index=False)
    panel.to_parquet(panel_path, index=False)
    cleaning.to_csv(PROCESSED / "cleaning_report.csv", index=False)
    coverage.to_csv(PROCESSED / "coverage_by_month.csv", index=False)
    name_matches.to_csv(PROCESSED / "dispatch_name_matches.csv", index=False)

    departure_gap = int(panel["departures"].sum() - len(trips))
    print(f"trips_clean.parquet rows: {len(trip_out):,}")
    print(f"stations.parquet rows: {len(stations):,}")
    print(f"station_hour.parquet rows: {len(panel):,}")
    print(f"Sum of departures minus kept trips (should be 0): {departure_gap}")
    print(
        "Weather coverage: "
        f"{panel['temperature_c'].notna().mean():.1%}; "
        "status coverage: "
        f"{panel['bikes_available'].notna().mean():.1%}"
    )
    july4 = panel.loc[panel["is_semiquincentennial"], "hour"]
    fifa_days = panel.loc[panel["is_fifa_match_day"], "hour"].dt.date.nunique()
    print(f"FIFA match days present in the trip window: {fifa_days}")
    print(f"July 4, 2026 station-hours in the panel: {len(july4):,}")
    print("Monthly coverage saved to coverage_by_month.csv; EDA plots the coverage over time.")
    if departure_gap != 0:
        raise RuntimeError("Departure totals do not match the cleaned trip file.")

## Build and save the data

This downloads source files when needed. Published data may now extend beyond the supplied June 2026 snapshot. Reuse the supplied processed files to reproduce that snapshot.

In [11]:
main()

Project: /content/drive/MyDrive/Milestone II

  trips  indego-trips-2026-q2.zip
  trips  indego-trips-2026-q1.zip
  trips  indego-trips-2025-q4.zip
  trips  indego-trips-2025-q3.zip
  trips  indego-trips-2025-q2.zip
  trips  indego-trips-2025-q1.zip
  trips  indego-trips-2024-q4.zip
  trips  indego-trips-2024-q3.zip
  trips  indego-trips-2024-q2.zip
  trips  indego-trips-2024-q1.zip
  stations  indego-stations-2026-10-15.csv
  station context
  situations 2024
  weather 2024
  sports 2024
  dispatch 2024
  station status 2024: 6 months
  situations 2025
  weather 2025
  sports 2025
  dispatch 2025
  station status 2025: 12 months
  situations 2026
  weather 2026
  sports 2026
  dispatch 2026
  station status 2026: 7 months

Reading and cleaning Indego trips
                               step    rows
                          rows_read 3316798
             dropped_bad_start_time       0
      dropped_missing_start_station       0
             dropped_duplicate_trip       0
     dropped

/tmp/ipykernel_1037/3947907376.py:119: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  panel["sports_any"] = panel["sports_any"].fillna(False).astype("boolean")



Saving tables
trips_clean.parquet rows: 3,316,664
stations.parquet rows: 385
station_hour.parquet rows: 5,950,039
Sum of departures minus kept trips (should be 0): 0
Weather coverage: 100.0%; status coverage: 26.5%
FIFA match days present in the trip window: 5
July 4, 2026 station-hours in the panel: 0
Monthly coverage saved to coverage_by_month.csv; EDA plots the coverage over time.
